# C6-pytorch — Practice p16 — Solution


The experiment snapshots the actual stored tensors, performs exactly
200 module calls, and compares both the storage and endpoint outputs.
The corrected module keeps identical values and arithmetic while
making the frozen intent explicit.


In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)
np.random.seed(20260804)

class HandBuilt(nn.Module):
    """The teammate's module — do not edit; audit as-is."""

    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.tensor([[1.0, -2.0], [0.5, 1.5]]))
        self.bias = nn.Parameter(torch.tensor([0.25, -0.75]))

    def forward(self, x):
        return x @ self.weight.T + self.bias


torch.manual_seed(20260804)
xs = torch.randn(64, 2)

hb = HandBuilt()
flags_before = sorted(p.requires_grad for p in hb.parameters())
snapshots = tuple(p.clone() for p in hb.parameters())
first_output = hb(xs)
for _ in range(198):
    hb(xs)
last_output = hb(xs)
params_unchanged = all(torch.equal(p, s) for p, s in zip(hb.parameters(), snapshots))
outputs_same = bool(torch.equal(first_output, last_output))

class FrozenHandBuilt(nn.Module):
    def __init__(self, weight, bias):
        super().__init__()
        self.weight = nn.Parameter(weight.clone(), requires_grad=False)
        self.bias = nn.Parameter(bias.clone(), requires_grad=False)

    def forward(self, x):
        return x @ self.weight.T + self.bias


calm = FrozenHandBuilt(hb.weight, hb.bias)
flags_after = sorted(p.requires_grad for p in calm.parameters())
same_map = bool((calm(xs) == hb(xs)).all())

flags_before, params_unchanged, outputs_same, flags_after, same_map


`params_unchanged` and `outputs_same` show that repeated forward calls
neither rewrite stored parameters nor change this deterministic map.
The flag controls derivative tracking, not automatic mutation; the
course still freezes inference-only parameters to document intent and
avoid building needless derivative bookkeeping.


### Answer check


In [ ]:
assert flags_before == [True, True]
assert params_unchanged is True
assert outputs_same is True
assert flags_after == [False, False]
assert same_map is True
assert torch.equal(calm.weight, snapshots[0]) and torch.equal(calm.bias, snapshots[1])
